In [1]:
# NLP + K-Means Clustering for `au_courts.csv`

## 1. Install Required Libraries

```python
# Run this cell if the packages are not already installed

!pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud
```

---

## 2. Import Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

from wordcloud import WordCloud

import warnings
warnings.filterwarnings("ignore")
```

---

## 3. Download NLTK Resources

```python
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
```

---

## 4. Load the Dataset

Make sure `au_courts.csv` is in the same directory as your Jupyter Notebook.

```python
df = pd.read_csv("au_courts.csv")

print("Dataset shape:", df.shape)
df.head()
```

---

## 5. Inspect the Dataset

```python
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()
```

Check for missing values:

```python
print(df.isnull().sum().sort_values(ascending=False))
```

Check a sample of the data:

```python
display(df.head(10))
```

---

## 6. Automatically Find Text Columns

The following code identifies columns containing text.

```python
text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Text columns:")
print(text_columns)
```

Check the amount of text in each column:

```python
for col in text_columns:
    avg_length = df[col].fillna("").astype(str).str.len().mean()
    print(f"{col}: average character length = {avg_length:.2f}")
```

---

## 7. Select the Main Text Column

If you know the column containing the court/judgment text, specify it here.

For example:

```python
TEXT_COLUMN = "text"
```

If your dataset uses a different name, change `"text"` to the appropriate column.

To see the available columns:

```python
print(df.columns.tolist())
```

### Automatic option

If you do not know which column contains the main text, this code selects the object column with the largest average text length:

```python
text_lengths = {}

for col in text_columns:
    text_lengths[col] = df[col].fillna("").astype(str).str.len().mean()

TEXT_COLUMN = max(text_lengths, key=text_lengths.get)

print("Selected text column:", TEXT_COLUMN)
```

---

## 8. Remove Rows with Missing Text

```python
df = df.dropna(subset=[TEXT_COLUMN]).copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

print("Dataset shape after removing missing text:", df.shape)
```

---

# 9. NLP Text Preprocessing

The preprocessing performs:

* Lowercase conversion
* Removal of URLs
* Removal of punctuation
* Removal of numbers
* Stop-word removal
* Lemmatization

```python
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Keep alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenization
    words = text.split()
    
    # Stopword removal and lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(words)
```

Apply the preprocessing:

```python
df["clean_text"] = df[TEXT_COLUMN].apply(preprocess_text)

df[[TEXT_COLUMN, "clean_text"]].head()
```

---

# 10. Check the Processed Text

```python
for i in range(min(5, len(df))):
    print("Original:")
    print(df[TEXT_COLUMN].iloc[i][:500])
    
    print("\nProcessed:")
    print(df["clean_text"].iloc[i][:500])
    
    print("\n" + "="*80 + "\n")
```

---

# 11. Remove Empty Documents

Some documents may contain only stopwords or punctuation.

```python
df = df[df["clean_text"].str.strip() != ""].copy()

print("Dataset shape:", df.shape)
```

---

# 12. Convert Text to TF-IDF Features

TF-IDF converts the court documents into numerical features that K-Means can process.

```python
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)
```

---

# 13. Examine Important TF-IDF Terms

```python
feature_names = vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFirst 100 features:")
print(feature_names[:100])
```

---

# 14. Find the Best Number of K-Means Clusters

We can test several values of `k`.

```python
k_values = range(2, 11)

inertia = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))

print("Finished testing cluster numbers.")
```

---

# 15. Elbow Method

```python
plt.figure(figsize=(10, 6))

plt.plot(k_values, inertia, marker="o")

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Selecting k")

plt.grid(True)
plt.show()
```

The point where the reduction in inertia starts to slow down is commonly used as an indication of a suitable `k`.

---

# 16. Silhouette Score

```python
plt.figure(figsize=(10, 6))

plt.plot(k_values, silhouette_scores, marker="o")

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different k Values")

plt.grid(True)
plt.show()
```

Print the scores:

```python
for k, score in zip(k_values, silhouette_scores):
    print(f"k = {k}: silhouette score = {score:.4f}")
```

---

# 17. Choose the Number of Clusters

For an initial experiment, we can select the `k` with the highest silhouette score.

```python
best_k = k_values[np.argmax(silhouette_scores)]

print("Selected number of clusters:", best_k)
```

You can also manually choose a value based on the elbow plot:

```python
# Example:
# best_k = 5
```

---

# 18. Run K-Means

```python
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print(df["cluster"].value_counts().sort_index())
```

---

# 19. View Cluster Distribution

```python
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="cluster",
    order=sorted(df["cluster"].unique())
)

plt.title("Number of Court Documents per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Documents")

plt.show()
```

---

# 20. Find the Most Important Words in Each Cluster

The K-Means cluster centers show which TF-IDF words are most strongly associated with each cluster.

```python
terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_num in range(best_k):
    top_terms = [
        terms[ind]
        for ind in order_centroids[cluster_num, :20]
    ]
    
    print(f"\nCluster {cluster_num}")
    print("-" * 50)
    print(", ".join(top_terms))
```

---

# 21. Create a Cluster Summary Table

```python
cluster_summary = []

for cluster_num in range(best_k):
    top_terms = [
        terms[ind]
        for ind in order_centroids[cluster_num, :15]
    ]
    
    cluster_summary.append({
        "Cluster": cluster_num,
        "Document_Count": int((df["cluster"] == cluster_num).sum()),
        "Top_Terms": ", ".join(top_terms)
    })

cluster_summary = pd.DataFrame(cluster_summary)

display(cluster_summary)
```

---

# 22. View Example Court Documents from Each Cluster

```python
for cluster_num in range(best_k):
    
    print("\n" + "="*100)
    print(f"CLUSTER {cluster_num}")
    print("="*100)
    
    cluster_docs = df[df["cluster"] == cluster_num].head(3)
    
    for idx, row in cluster_docs.iterrows():
        print("\nDocument index:", idx)
        print(row[TEXT_COLUMN][:1000])
        print("-"*80)
```

---

# 23. Word Clouds for Each Cluster

```python
for cluster_num in range(best_k):
    
    cluster_text = " ".join(
        df.loc[df["cluster"] == cluster_num, "clean_text"]
    )
    
    if not cluster_text.strip():
        continue
    
    wordcloud = WordCloud(
        width=1000,
        height=500,
        background_color="white",
        max_words=100
    ).generate(cluster_text)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud - Cluster {cluster_num}")
    plt.show()
```

---

# 24. Visualise the Clusters Using PCA/SVD

TF-IDF can contain thousands of dimensions, so we reduce it to two dimensions for visualisation.

`TruncatedSVD` works well with sparse TF-IDF matrices.

```python
svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

print("Reduced matrix shape:", X_2d.shape)
```

Create the plot:

```python
plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.6
)

plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")
plt.title("K-Means Clusters of Australian Court Documents")

plt.colorbar(scatter, label="Cluster")

plt.show()
```

---

# 25. Add the Cluster Labels to the Original Dataset

The cluster assignment is now stored in:

```python
df["cluster"]
```

Check the result:

```python
display(
    df[[TEXT_COLUMN, "cluster"]].head(20)
)
```

---

# 26. Sort Documents by Cluster

```python
clustered_data = df.sort_values("cluster")

display(clustered_data[[TEXT_COLUMN, "cluster"]].head(20))
```

---

# 27. Save the Results

Save the original data plus the NLP/K-Means cluster assignment.

```python
output_file = "au_courts_kmeans_clusters.csv"

df.to_csv(output_file, index=False)

print("Saved:", output_file)
```

---

# 28. Save the Cluster Summary

```python
cluster_summary.to_csv(
    "au_courts_cluster_summary.csv",
    index=False
)

print("Cluster summary saved.")
```

---

# 29. Final Results

```python
print("="*60)
print("K-MEANS NLP ANALYSIS COMPLETE")
print("="*60)

print("\nOriginal dataset size:")
print(len(df))

print("\nNumber of clusters:")
print(best_k)

print("\nSilhouette score:")
print(round(max(silhouette_scores), 4))

print("\nDocuments in each cluster:")
print(df["cluster"].value_counts().sort_index())

print("\nOutput files:")
print("1. au_courts_kmeans_clusters.csv")
print("2. au_courts_cluster_summary.csv")
```

---

## Optional: Automatically Generate Cluster Names

K-Means produces numerical labels such as Cluster 0, Cluster 1, etc. These labels do **not** inherently represent legal categories.

You can inspect the top terms and assign descriptive names manually.

```python
cluster_names = {}

for cluster_num in range(best_k):
    top_terms = [
        terms[ind]
        for ind in order_centroids[cluster_num, :10]
    ]
    
    print(f"Cluster {cluster_num}:")
    print(", ".join(top_terms))
    print()
```

After inspecting the terms, you could create labels such as:

```python
# Replace these with names appropriate to your actual results.

cluster_names = {
    0: "Cluster 0",
    1: "Cluster 1",
    2: "Cluster 2",
    3: "Cluster 3",
    4: "Cluster 4"
}

df["cluster_name"] = df["cluster"].map(cluster_names)
```

If the number of clusters is different, adjust the dictionary accordingly.

---

## Complete Workflow

The analysis follows this pipeline:

```text
au_courts.csv
      |
      v
Load Dataset
      |
      v
Identify Text Column
      |
      v
NLP Preprocessing
      |
      +--> Lowercase
      +--> Remove URLs
      +--> Remove punctuation
      +--> Remove stopwords
      +--> Lemmatization
      |
      v
TF-IDF Vectorisation
      |
      v
Find Optimal K
      |
      +--> Elbow Method
      +--> Silhouette Score
      |
      v
K-Means Clustering
      |
      v
Cluster Analysis
      |
      +--> Top Terms
      +--> Example Documents
      +--> Word Clouds
      +--> 2D Visualisation
      |
      v
Save Clustered Dataset
```

### Important note

The code deliberately **does not assume that a particular column is called `text`**. If `au_courts.csv` has a known judgment/document column, it is better to set `TEXT_COLUMN` explicitly rather than relying on the automatic selection.

Also, K-Means is an **unsupervised** method: the resulting clusters are groups of documents with similar TF-IDF representations, not automatically recognised legal categories.


SyntaxError: invalid decimal literal (3257599546.py, line 684)